# Sampling physical GAPMOE parameters with emcee

This notebook samples the current canonical physical parameters:

- `ML`: lens mass in solar masses
- `DL`: lens distance in pc
- `DS`: source distance in pc
- `mu_N`: heliocentric relative proper motion north in mas/yr
- `mu_E`: heliocentric relative proper motion east in mas/yr

The example first runs `PreRunner` to create event-local Genulens histogram files, then evaluates `GalacticModel.log_prob(...)` inside `emcee`.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
if (cwd / "src" / "gapmoe").exists():
    repo_root = cwd
elif (cwd.parent / "src" / "gapmoe").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError("Run this notebook from the repository root or example directory.")

sys.path.insert(0, str(repo_root / "src"))

import emcee
import matplotlib.pyplot as plt
import numpy as np

from gapmoe import GalacticModel, PhysicalParams, PreRunner

rng = np.random.default_rng(12345)
repo_root

## Create event-local histogram inputs

By default, `PreRunner` looks for Genulens at `../genulens`. The normal user inputs are the sky coordinates and optional source-selection settings. The distance grids are shared between `rho.dat` and `murel.dat` by default, so this example does not set `d_max_pc`, `dl_max_pc`, or `ds_max_pc` manually.


In [ ]:
genulens_root = (repo_root / ".." / "genulens").resolve()
if not (genulens_root / "pre_gapmoe").exists():
    raise FileNotFoundError(
        f"Genulens pre_gapmoe was not found at {genulens_root / 'pre_gapmoe'}. "
        "Set genulens_root to your local Genulens checkout."
    )

runner = PreRunner(
    genulens_root=genulens_root,
    output_dir=repo_root / "example" / "pre_runner_outputs",
)

pre_run = runner.run(
    ra_deg=270.0,
    dec_deg=-30.0,
    run_name="emcee_demo",
    seed=123,
)

pre_run.output_dir


In [ ]:
model = GalacticModel.from_pre_run(pre_run)

trial = PhysicalParams(ML=0.3, DL=5000.0, DS=8000.0, mu_N=5.0, mu_E=2.0)
model.log_prob(trial)

## Define the emcee target

This samples the Galactic prior itself. For a real event, add the light-curve likelihood to `log_probability`.

In [ ]:
bounds = {
    "ML": (0.08, 1.5),
    "DL": (100.0, 12000.0),
    "DS": (500.0, 16000.0),
    "mu_N": (-30.0, 30.0),
    "mu_E": (-30.0, 30.0),
}

def unpack(theta):
    ML, DL, DS, mu_N, mu_E = theta
    return PhysicalParams(ML=ML, DL=DL, DS=DS, mu_N=mu_N, mu_E=mu_E)

def in_bounds(theta):
    ML, DL, DS, mu_N, mu_E = theta
    return (
        bounds["ML"][0] <= ML <= bounds["ML"][1]
        and bounds["DL"][0] <= DL <= bounds["DL"][1]
        and bounds["DS"][0] <= DS <= bounds["DS"][1]
        and bounds["mu_N"][0] <= mu_N <= bounds["mu_N"][1]
        and bounds["mu_E"][0] <= mu_E <= bounds["mu_E"][1]
        and DS > DL
    )

def log_probability(theta):
    if not in_bounds(theta):
        return -np.inf
    return model.log_prob(unpack(theta))

In [ ]:
def draw_finite_initial_positions(nwalkers, max_tries=100000):
    positions = []
    while len(positions) < nwalkers and max_tries > 0:
        max_tries -= 1
        theta = np.array([
            rng.uniform(*bounds["ML"]),
            rng.uniform(*bounds["DL"]),
            rng.uniform(*bounds["DS"]),
            rng.uniform(*bounds["mu_N"]),
            rng.uniform(*bounds["mu_E"]),
        ])
        lp = log_probability(theta)
        if np.isfinite(lp):
            positions.append(theta)
    if len(positions) != nwalkers:
        raise RuntimeError("Could not find enough finite initial walker positions.")
    return np.array(positions)

ndim = 5
nwalkers = 32
initial = draw_finite_initial_positions(nwalkers)
initial[:3]

In [ ]:
sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability)
sampler.run_mcmc(initial, 1000, progress=True)

acceptance_fraction = np.mean(sampler.acceptance_fraction)
acceptance_fraction

In [ ]:
chain = sampler.get_chain(discard=200, thin=5, flat=True)
labels = ["ML", "DL", "DS", "mu_N", "mu_E"]

for i, label in enumerate(labels):
    q16, q50, q84 = np.percentile(chain[:, i], [16, 50, 84])
    print(f"{label:>4s}: {q50:.4g} -{q50 - q16:.4g} +{q84 - q50:.4g}")

In [ ]:
fig, axes = plt.subplots(ndim, 1, figsize=(9, 7), sharex=True)
raw_chain = sampler.get_chain()

for i, ax in enumerate(axes):
    ax.plot(raw_chain[:, :, i], color="black", alpha=0.2, lw=0.5)
    ax.set_ylabel(labels[i])

axes[-1].set_xlabel("step")
fig.tight_layout()